In [ ]:
!pip install faiss-cpu transformers datasets torch

In [66]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import os
from typing import Optional, List, Tuple
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
import faiss

In [67]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [68]:
class Data:
    def __init__(self, seq_len: int = 32, seed: int = 42):
        rng = np.random.RandomState(seed)

        # LLM generated templates
        templates = [
            "{person} walked through the {adj} {place} and felt {emotion}.",
            "the {adj} {animal} rested beneath the {adj2} {plant}.",
            "{person} watched the {animal} near the {adj} {place}.",
            "a {adj} wind blew across the {place} at {time}.",
            "the {plant} grew tall beside the {adj2} {place}.",
            "{person} found a {adj} {object} inside the {adj2} {place}.",
            "the {animal} ran quickly past the {adj} {plant}.",
            "at {time} the {adj} sky hung over the {place}.",
            "{person} carried the {adj2} {object} toward the {place}.",
            "a {animal} sat quietly on the {adj} {object}.",
        ]

        words = {
            "person":  ["alice", "bob", "clara", "dan", "eve", "frank",
                        "grace", "henry", "iris", "jack"],
            "animal":  ["fox", "owl", "deer", "wolf", "crow", "hare",
                        "hawk", "bear", "frog", "moth"],
            "place":   ["forest", "canyon", "meadow", "village", "harbor",
                        "bridge", "tower", "chapel", "market", "garden"],
            "plant":   ["willow", "cedar", "fern", "ivy", "maple",
                        "birch", "elm", "oak", "pine", "moss"],
            "adj":     ["silent", "golden", "ancient", "narrow", "frozen",
                        "dusty", "hollow", "crimson", "faint", "vast"],
            "adj2":    ["small", "broken", "hidden", "bright", "dark",
                        "stone", "wooden", "iron", "glass", "worn"],
            "object":  ["lantern", "mirror", "compass", "journal", "key",
                        "basket", "clock", "ring", "map", "coin"],
            "emotion": ["wonder", "calm", "sorrow", "joy", "unease",
                        "relief", "longing", "pride", "doubt", "peace"],
            "time":    ["dawn", "dusk", "midnight", "noon", "sunrise",
                        "sunset", "twilight", "daybreak", "evening", "morning"],
        }

        sentences = []
        for _ in range(3000):
            tmpl = rng.choice(templates)
            filled = tmpl
            for slot, pool in words.items():
                while "{" + slot + "}" in filled:
                    filled = filled.replace("{" + slot + "}", rng.choice(pool), 1)
            sentences.append(filled)

        corpus = " ".join(sentences).lower()
        all_words = corpus.split()

        self.vocab = sorted(set(all_words))
        self.vocab_size = len(self.vocab)
        self.word2id = {w: i for i, w in enumerate(self.vocab)}
        self.id2word = {i: w for i, w in enumerate(self.vocab)}

        token_ids = [self.word2id[w] for w in all_words]

        stride = seq_len // 2
        sequences = []
        for start in range(0, len(token_ids) - seq_len, stride):
            sequences.append(token_ids[start : start + seq_len + 1])

        self.data = torch.tensor(sequences, dtype=torch.long)
        print(
            f"Dataset: {len(self.data)} sequences | "
            f"vocab size: {self.vocab_size} | "
            f"total words: {len(all_words)}"
        )


In [69]:
SEQ_LEN = 32
BATCH_SIZE = 8
dataset = Data(seq_len=SEQ_LEN)
dataloader = DataLoader(dataset.data, batch_size=BATCH_SIZE, shuffle=True)

Dataset: 1574 sequences | vocab size: 160 | total words: 25206


In [70]:
class Attention(nn.Module):
    # casual mask (scaled dp)
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # Project to Q, K, V at once
        qkv = self.qkv_proj(x)                            # (B, T, 3*C)
        q, k, v = qkv.chunk(3, dim=-1)                    # each (B, T, C)

        # reshape to heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, H, T, D)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scale = math.sqrt(self.head_dim)
        scores = (q @ k.transpose(-2, -1)) / scale        # (B, H, T, T)

        # mask
        causal_mask = torch.tril(torch.ones(T, T, device=x.device))
        scores = scores.masked_fill(causal_mask == 0, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(out)

In [71]:
class FeedForward(nn.Module):

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [72]:
class Transformer(nn.Module):

    def __init__(self, d_model: int, n_heads: int, d_ff: int, capture_context: bool = False):
        super().__init__()
        self.attn = Attention(d_model, n_heads)
        self.ffn  = FeedForward(d_model, d_ff)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)

        self.capture_context = capture_context
        self.captured: Optional[torch.Tensor] = None   

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))

        # save the input that is about to enter the FFN
        ffn_input = self.ln2(x)
        if self.capture_context:
            self.captured = ffn_input.detach()
            
        x = x + self.ffn(ffn_input)
        return x

In [73]:
class TransformerLM(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 3,
        d_ff: int = 128,
        max_seq_len: int = 512,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)

        # stack transformer blocks
        self.blocks = nn.ModuleList([
            Transformer(
                d_model, n_heads, d_ff,
                capture_context=(i == n_layers - 1)   # last block
            )
            for i in range(n_layers)
        ])

        self.ln_final = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)    

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device).unsqueeze(0)

        x = self.tok_emb(idx) + self.pos_emb(positions)

        for block in self.blocks:
            x = block(x)

        x = self.ln_final(x)
        return self.head(x)        

    @property
    def context_vectors(self) -> Optional[torch.Tensor]:
        # to return from the last block's forward pass
        return self.blocks[-1].captured

In [74]:
D_MODEL  = 64
N_HEADS  = 4
N_LAYERS = 3
D_FF     = 128

transformer = TransformerLM(
    vocab_size=dataset.vocab_size,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,).to(DEVICE)

n_params = sum(p.numel() for p in transformer.parameters())
n_params

153376

In [75]:
def train(
    model: TransformerLM,
    loader: DataLoader,
    epochs: int = 15,
    lr: float = 1e-3,
    device: torch.device = DEVICE,):

    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = []

    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        n_batches  = 0

        for batch in loader:
            inputs  = batch[:, :-1].to(device)   # (B, T)
            targets = batch[:, 1:].to(device)    # (B, T)

            optimizer.zero_grad()
            logits = model(inputs)               # (B, T, V)
            loss = criterion(
                logits.reshape(-1, model.vocab_size),
                targets.reshape(-1),
            )
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

        avg = total_loss / n_batches
        history.append(avg)
        print(f"  epoch {epoch:>2} —  loss: {avg:.4f}")

    return history

In [76]:
losshistory = train(transformer, dataloader, epochs=20)

  epoch  1 —  loss: 2.9641
  epoch  2 —  loss: 1.7431
  epoch  3 —  loss: 1.3985
  epoch  4 —  loss: 1.3221
  epoch  5 —  loss: 1.2899
  epoch  6 —  loss: 1.2678
  epoch  7 —  loss: 1.2467
  epoch  8 —  loss: 1.2224
  epoch  9 —  loss: 1.1972
  epoch 10 —  loss: 1.1632
  epoch 11 —  loss: 1.1315
  epoch 12 —  loss: 1.0898
  epoch 13 —  loss: 1.0483
  epoch 14 —  loss: 1.0087
  epoch 15 —  loss: 0.9586
  epoch 16 —  loss: 0.9152
  epoch 17 —  loss: 0.8686
  epoch 18 —  loss: 0.8257
  epoch 19 —  loss: 0.7778
  epoch 20 —  loss: 0.7325


In [77]:
def datastore(
    model: TransformerLM,
    loader: DataLoader,
    device: torch.device = DEVICE,
) -> Tuple[np.ndarray, np.ndarray, faiss.IndexFlatL2]:

# run model over training set, get all pairs and build faiss index

    model.eval()
    all_keys, all_vals = [], []

    with torch.no_grad():
        for batch in loader:
            inputs  = batch[:, :-1].to(device)
            targets = batch[:, 1:].to(device)

            _ = model(inputs)                    
            ctx = model.context_vectors        

            all_keys.append(ctx.view(-1, model.d_model).cpu().numpy())
            all_vals.append(targets.reshape(-1).cpu().numpy())

    keys = np.vstack(all_keys).astype(np.float32)
    vals = np.hstack(all_vals).astype(np.int32)

    index = faiss.IndexFlatL2(model.d_model)
    index.add(keys)

    print(f"{keys.shape[0]:,} entries, d={keys.shape[1]}")
    return keys, vals, index

In [78]:
keys, vals, faiss_index = datastore(transformer, dataloader)

50,368 entries, d=64


In [79]:
class KNNLM:
    # wrap transformerlm and datastore to get token preds."

    def __init__(
        self,
        model: TransformerLM,
        keys: np.ndarray,
        vals: np.ndarray,
        index: faiss.IndexFlatL2,
        k: int = 8,
        temperature: float = 10.0,
    ):
        self.model = model
        self.keys  = keys
        self.vals  = vals
        self.index = index
        self.k     = k
        self.temperature = temperature  

    def _knn_probs(self, queries: torch.Tensor) -> torch.Tensor:
        B, T, D = queries.shape
        flat = queries.reshape(-1, D).cpu().numpy().astype(np.float32)

        distances, indices = self.index.search(flat, self.k)      

        neighbour_tokens = torch.tensor(
            self.vals[indices], dtype=torch.long
        ) 

        neg_dists = -torch.tensor(distances, dtype=torch.float32) / self.temperature
        weights = F.softmax(neg_dists, dim=-1)                   

        V = self.model.vocab_size
        probs = torch.zeros(flat.shape[0], V)
        probs.scatter_add_(1, neighbour_tokens, weights)

        return probs.view(B, T, V)

    def forward(
        self,
        input_ids: torch.Tensor,
        lam: float = 0.25,) -> Tuple[torch.Tensor, torch.Tensor]:

        lm_logits = self.model(input_ids)             
        lm_probs  = F.softmax(lm_logits, dim=-1)

        ctx = self.model.context_vectors               
        knn_probs = self._knn_probs(ctx).to(lm_probs.device)

        combined = lam * knn_probs + (1 - lam) * lm_probs
        combined_logits = torch.log(combined + 1e-10)

        return combined_logits, lm_logits

In [80]:
K = 8 
knn_lm = KNNLM(
    model=transformer,
    keys=keys,
    vals=vals,
    index=faiss_index,
    k=K,
    temperature=10.0,
)

In [81]:
@torch.no_grad()
def generate(
    forward_fn,
    dataset: Data,
    prompt: str = "the sun",
    max_new_tokens: int = 20,
    temperature: float = 0.8,
    device: torch.device = DEVICE,
) -> str:

    tokens = prompt.lower().split()
    ids = [dataset.word2id.get(w, 0) for w in tokens]
    generated = torch.tensor([ids], device=device)

    for _ in range(max_new_tokens):
        out = forward_fn(generated)
        logits = out[0] if isinstance(out, tuple) else out

        next_logits = logits[0, -1, :] / temperature
        probs = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        generated = torch.cat([generated, next_id.unsqueeze(0)], dim=1)

    words = [dataset.id2word.get(i, "<UNK>") for i in generated[0].cpu().tolist()]
    return " ".join(words)

In [83]:
for prompt in ["alice walked through", "the golden fox"]:
    print(f"\ninput: \"{prompt}\"\n")

    base_text = generate(transformer, dataset, prompt=prompt)
    print(f"  Base LM : {base_text}")

    knn_text = generate(
        lambda x: knn_lm.forward(x, lam=0.3),
        dataset,
        prompt=prompt,
    )
    print(f"  kNN-LM  : {knn_text}")


input: "alice walked through"

  Base LM : alice walked through the frozen village and felt pride. iris watched the owl near the faint market. at evening the the silent sky
  kNN-LM  : alice walked through the narrow tower and felt doubt. at noon the silent sky hung over the harbor. the moss grew tall beside

input: "the golden fox"

  Base LM : the golden fox rested beneath the hidden pine. the moth ran quickly past the vast maple. the faint grew tall beside the worn
  kNN-LM  : the golden fox rested beneath the stone pine. alice walked through the frozen tower and felt doubt. the willow grew tall beside the
